# Discontinued Music Player Price Analysis

**Data Studio Project 2**

Looking at whether discontinued portable music players are selling for more than their original retail price. The hypothesis: nostalgia + people wanting dedicated music players (no notifications, no distractions) has driven prices up.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')

## MSRP Baseline Data

What these things cost when they were new. Sources: Apple press releases, Wikipedia, Microsoft archives.

In [ ]:
msrp_df = pd.read_csv('mrsp_baseline_csv - Sheet1.csv')
msrp_df['Discontinuation_Date'] = pd.to_datetime(msrp_df['Discontinuation_Date'])
msrp_df['Years_Since_Discontinued'] = ((datetime.now() - msrp_df['Discontinuation_Date']).dt.days / 365).round(1)
msrp_df

In [ ]:
# timeline viz - when was each device discontinued?
sorted_df = msrp_df.sort_values('Years_Since_Discontinued', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(sorted_df['Device'], sorted_df['Years_Since_Discontinued'], color='#2c3e50', edgecolor='none')
ax.set_xlabel('Years Since Discontinued', fontsize=11)
ax.set_title('Time Off Market by Device', fontsize=13, fontweight='bold', pad=12)

# add MSRP labels
for i, (device, years, msrp) in enumerate(zip(sorted_df['Device'], sorted_df['Years_Since_Discontinued'], sorted_df['Last_MSRP_USD'])):
    ax.text(years + 0.2, i, f'${msrp:,.0f}', va='center', fontsize=9, color='#555')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', length=0)

plt.tight_layout()
plt.show()

## Preliminary eBay Research

Before pulling full API data, I did some quick searches to validate that the price appreciation story is real.

In [ ]:
# Load eBay Browse API data (live current listings)
api_df = pd.read_csv('ebay_browse_api_data.csv')

# Handle column name variations
if 'vs_MSRP_Pct' not in api_df.columns and 'Median_vs_MSRP' in api_df.columns:
    api_df['vs_MSRP_Pct'] = api_df['Median_vs_MSRP']

api_df = api_df.sort_values('vs_MSRP_Pct', ascending=False)
print("LIVE eBay DATA from Browse API")
print("Condition: Used - Good or better")
print("Note: These are current ASKING prices, not completed sales")
print()
api_df

In [ ]:
# MSRP vs eBay Median Price
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(api_df))
width = 0.35

bars1 = ax.bar(x - width/2, api_df['Original_MSRP'], width, label='Original MSRP', color='#95a5a6', edgecolor='none')
bars2 = ax.bar(x + width/2, api_df['eBay_Median'], width, label='eBay Median Price', color='#2c3e50', edgecolor='none')

ax.set_ylabel('Price ($)', fontsize=11)
ax.set_title('Original MSRP vs Current eBay Prices', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(api_df['Device'], rotation=35, ha='right', fontsize=9)
ax.legend(frameon=False, fontsize=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

In [ ]:
# Price change from MSRP
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#27ae60' if x > 0 else '#c0392b' for x in api_df['vs_MSRP_Pct']]
ax.barh(api_df['Device'], api_df['vs_MSRP_Pct'], color=colors, edgecolor='none')
ax.axvline(x=0, color='#2c3e50', linewidth=1.5)

ax.set_xlabel('% Change from Original MSRP', fontsize=11)
ax.set_title('Current eBay Price vs Original MSRP', fontsize=13, fontweight='bold', pad=12)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', length=0)

for i, (device, pct) in enumerate(zip(api_df['Device'], api_df['vs_MSRP_Pct'])):
    offset = 3 if pct >= 0 else -3
    ha = 'left' if pct >= 0 else 'right'
    ax.text(pct + offset, i, f'{pct:+.0f}%', va='center', ha=ha, fontsize=9, fontweight='bold',
            color='#27ae60' if pct > 0 else '#c0392b')

plt.tight_layout()
plt.show()

## Key Findings (from 1,346 eBay Listings)

**The Surprise Winner:**
- **iPod Shuffle 4th Gen: +39%** — The only device selling ABOVE original MSRP
- At $49 original, now median $68. The "dumbest" iPod is the most valuable.

**The Collector Story (iPod 1st Gen):**
- Median: $75 (most are damaged/parts)
- Mean: **$1,647** — Sealed/mint units pull the average way up
- Range: $25 to thousands — condition is everything

**The Losers:**
- iPod Mini: **-80%** — Replaced by Nano, forgotten
- iPod Touch: **-67%** — Too recent, too similar to iPhone
- Zune HD: **-40%** — Cult following ≠ market value

**The Story:**
Most discontinued devices depreciate. The exception? Devices that are either:
1. **So basic they're charming** (Shuffle)
2. **So rare and pristine they're collectible** (1st Gen sealed)

The "nostalgia premium" only exists at the extremes.

## The Story

> "The nostalgia economy is real—but it's picky. Most discontinued tech loses value the moment it's off the shelf. The exceptions are devices that hit a sweet spot: basic enough to be charming, rare enough to be collectible, or pristine enough to be an artifact."

**The iPod Shuffle paradox:** The simplest, "dumbest" iPod is the only one appreciating. No screen, no clicks, just music. In an era of infinite distraction, that simplicity now commands a premium.

**The 1st Gen collector trap:** The original iPod can sell for thousands—but only sealed, mint, in-box. The median is $75 because most are scratched relics. Condition is the entire game.

**The forgotten middle:** iPod Mini, Touch, Nano—devices that were "good enough" but not iconic. They depreciate like any other used electronics.

## eBay API Integration

Pulling current listing data from eBay's Browse API with consistent methodology:
- **100 listings per device** (equal sample sizes)
- **No condition filter** (includes all conditions from "For Parts" to "New/Sealed")
- **Data type:** Current asking prices (not sold prices)

This approach reveals the full price spectrum—showing both the depreciation of average units AND the appreciation of premium/sealed units.

In [ ]:
import requests
import base64
import os
from dotenv import load_dotenv

# Load credentials from .env file
load_dotenv('ebay-verification/.env')

APP_ID = os.getenv('EBAY_APP_ID')
CERT_ID = os.getenv('EBAY_CERT_ID')

def get_oauth_token():
    """Get OAuth token from eBay using Client Credentials flow"""
    url = "https://api.ebay.com/identity/v1/oauth2/token"
    
    credentials = f"{APP_ID}:{CERT_ID}"
    encoded_credentials = base64.b64encode(credentials.encode()).decode()
    
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {encoded_credentials}"
    }
    
    data = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope"
    }
    
    response = requests.post(url, headers=headers, data=data)
    
    if response.status_code == 200:
        return response.json()['access_token']
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
        return None

token = get_oauth_token()
print("Token obtained!" if token else "Failed to get token")

In [ ]:
TARGET_LISTINGS = 100  # Equal listings per device

def search_ebay_all(query, target=100):
    """Search eBay Browse API for ALL conditions (no filter)"""
    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"
    
    headers = {
        "Authorization": f"Bearer {token}",
        "X-EBAY-C-MARKETPLACE-ID": "EBAY_US"
    }
    
    all_items = []
    offset = 0
    
    while len(all_items) < target:
        params = {
            "q": query,
            "limit": min(50, target - len(all_items)),
            "offset": offset
        }
        
        response = requests.get(url, headers=headers, params=params)
        
        if response.status_code == 200:
            data = response.json()
            items = data.get('itemSummaries', [])
            if not items:
                break
            all_items.extend(items)
            offset += len(items)
            if len(items) < 50:  # No more results available
                break
        else:
            print(f"Error: {response.status_code}")
            break
    
    return all_items[:target]

# Device queries
device_queries = {
    'iPod Classic': 'iPod Classic 160GB',
    'iPod Mini': 'iPod Mini',
    'iPod Nano': 'iPod Nano 7th generation',
    'iPod Shuffle': 'iPod Shuffle 4th generation',
    'iPod Touch': 'iPod Touch 7th generation',
    'iPod 1st Gen': 'iPod 1st generation original 2001',
    'iPod Hi-Fi': 'Apple iPod Hi-Fi',
    'Zune HD': 'Microsoft Zune HD',
    'Sony Walkman': 'Sony Walkman NW-A55',
    'Mac Mini M1': 'Apple Mac Mini M1'
}

# Fetch ALL listings (no condition filter)
print("Fetching eBay listings (ALL conditions - includes New/Sealed)...")
print("-" * 50)
all_results = {}
for device, query in device_queries.items():
    print(f"Searching: {device}...", end=" ")
    items = search_ebay_all(query, TARGET_LISTINGS)
    all_results[device] = items
    print(f"Found {len(items)} listings")

print("-" * 50)
print(f"Total listings: {sum(len(v) for v in all_results.values())}")
print("Condition filter: NONE (all conditions included)")

In [ ]:
# Analyze prices and save to CSV
price_summary = []
all_listings_data = []

# MSRP lookup
msrp_lookup = {
    'iPod Classic': 249,
    'iPod Mini': 249,
    'iPod Nano': 149,
    'iPod Shuffle': 49,
    'iPod Touch': 399,
    'iPod 1st Gen': 399,
    'iPod Hi-Fi': 349,
    'Zune HD': 290,
    'Sony Walkman': 298,
    'Mac Mini M1': 699
}

for device, listings in all_results.items():
    if not listings:
        continue
    
    prices = []
    for item in listings:
        if 'price' in item:
            price = float(item['price']['value'])
            prices.append(price)
            
            # Save individual listing
            all_listings_data.append({
                'Device_Category': device,
                'Title': item.get('title', ''),
                'Price': price,
                'Condition': item.get('condition', 'Unknown'),
                'Item_URL': item.get('itemWebUrl', '')
            })
    
    if prices:
        msrp_val = msrp_lookup.get(device)
        sorted_prices = sorted(prices, reverse=True)
        top_25_idx = len(sorted_prices) // 4
        top_25_price = np.mean(sorted_prices[:max(1, top_25_idx)])
        
        price_summary.append({
            'Device': device,
            'Original_MSRP': msrp_val,
            'eBay_Low': round(min(prices), 2),
            'eBay_High': round(max(prices), 2),
            'eBay_Median': round(np.median(prices), 2),
            'eBay_Mean': round(np.mean(prices), 2),
            'Top_25_Pct_Price': round(top_25_price, 2),
            'Listings_Count': len(prices),
            'Median_vs_MSRP': round((np.median(prices) - msrp_val) / msrp_val * 100, 1),
            'Top25_vs_MSRP': round((top_25_price - msrp_val) / msrp_val * 100, 1),
            'Source': 'eBay Browse API'
        })

# Create DataFrames
price_df = pd.DataFrame(price_summary)
price_df = price_df.sort_values('Top25_vs_MSRP', ascending=False)

listings_df = pd.DataFrame(all_listings_data)

# Save to CSV
price_df.to_csv('ebay_browse_api_data.csv', index=False)
listings_df.to_csv('ebay_all_listings.csv', index=False)

print("DATA SAVED TO CSV FILES")
print("=" * 60)
print(f"Summary: ebay_browse_api_data.csv ({len(price_df)} devices)")
print(f"All listings: ebay_all_listings.csv ({len(listings_df)} listings)")
print("=" * 60)
print()
print("KEY COMPARISON: Median vs Top 25% Tier")
print("-" * 60)
price_df[['Device', 'Original_MSRP', 'eBay_Median', 'Median_vs_MSRP', 'Top_25_Pct_Price', 'Top25_vs_MSRP', 'eBay_High']]

### Price Distribution by Device

Individual charts showing price spread for each device from live eBay data.

In [ ]:
# Individual price distribution charts for each device
fig, axes = plt.subplots(4, 3, figsize=(14, 16))
axes = axes.flatten()

devices_with_data = [(d, l) for d, l in all_results.items() if l]

for idx, (device, listings) in enumerate(devices_with_data):
    if idx >= 12:
        break
    
    ax = axes[idx]
    prices = [float(item['price']['value']) for item in listings if 'price' in item]
    
    if prices:
        msrp = msrp_lookup.get(device, 0)
        
        # Histogram
        ax.hist(prices, bins=15, color='#3498db', edgecolor='white', alpha=0.7)
        
        # MSRP line
        if msrp:
            ax.axvline(x=msrp, color='#e74c3c', linestyle='--', linewidth=2, label=f'MSRP: ${msrp}')
        
        # Median line
        median_price = np.median(prices)
        ax.axvline(x=median_price, color='#27ae60', linestyle='-', linewidth=2, label=f'Median: ${median_price:.0f}')
        
        ax.set_title(device, fontsize=11, fontweight='bold')
        ax.set_xlabel('Price ($)', fontsize=9)
        ax.set_ylabel('Count', fontsize=9)
        ax.legend(fontsize=8)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

# Hide unused subplots
for idx in range(len(devices_with_data), 12):
    axes[idx].set_visible(False)

plt.suptitle('eBay Current Listing Prices by Device', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Summary comparison: MSRP vs Median vs Top 25% Asking Price
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(price_df))
width = 0.25

bars1 = ax.bar(x - width, price_df['Original_MSRP'], width, label='Original MSRP', color='#95a5a6', edgecolor='none')
bars2 = ax.bar(x, price_df['eBay_Median'], width, label='Median Asking', color='#e74c3c', edgecolor='none')
bars3 = ax.bar(x + width, price_df['Top_25_Pct_Price'], width, label='Top 25% Asking', color='#27ae60', edgecolor='none')

ax.set_ylabel('Price ($)', fontsize=11)
ax.set_title('Original MSRP vs Current eBay Prices (Median vs Premium Tier)', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(price_df['Device'], rotation=30, ha='right', fontsize=10)
ax.legend(frameon=False, fontsize=10)

# Add percentage labels for Top 25%
for i, (device, pct) in enumerate(zip(price_df['Device'], price_df['Top25_vs_MSRP'])):
    if pct is not None:
        color = '#27ae60' if pct > 0 else '#c0392b'
        ax.text(i + width, price_df['Top_25_Pct_Price'].iloc[i] + 15, f'{pct:+.0f}%', 
                ha='center', fontsize=9, color=color, fontweight='bold')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()